In [23]:
# Sam Brown
# sam_brown@mines.edu
# June 18
# Goal: Neural Net for predicting time until next event

# Directory
import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

# Imports
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('averages_events_2011-13')
df = df.iloc[1:-1] # account for shift to get mins until


In [25]:
# Define features and target
X = df[['mins_since', 'tide_change', 'tide_height', 'h_event', 'form_factor', 'slip_severity', 'total_delta', 'pre_slip_area']]
y = df['mins_until'].values.reshape(-1,1)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y , test_size = .2, random_state = 42)

# Standardize
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_scaled = scaler.fit_transform(y_train)
y_test_scaled = scaler.transform(y_test)

# Pytorch tensors 
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

In [27]:
# Neural Net
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(8, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16,1)

    def forward(self,x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x) # Regressing
        return x

In [29]:
model = Net()
criterion = nn.MSELoss() # Loss for regression
optimizer = optim.Adam(model.parameters(), lr = .01)


In [31]:
# Training loop
epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad() # Clears grad

    # Predictions and loss
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # Backprop
    loss.backward()

    # Update params
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [20/200], Loss: 0.9095
Epoch [40/200], Loss: 0.8113
Epoch [60/200], Loss: 0.7241
Epoch [80/200], Loss: 0.6645
Epoch [100/200], Loss: 0.6250
Epoch [120/200], Loss: 0.5895
Epoch [140/200], Loss: 0.5647
Epoch [160/200], Loss: 0.5404
Epoch [180/200], Loss: 0.5236
Epoch [200/200], Loss: 0.5078


In [33]:
# Eval
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor)
    y_pred = scaler.inverse_transform(y_pred_scaled.numpy())
    y_test_orig = scaler.inverse_transform(y_test_tensor.numpy())

# predictions vs actual
print("\nPredictions vs Actual:")
for pred, actual in zip(y_pred.flatten(), y_test_orig.flatten()):
    print(f"Predicted: {pred:.2f}, Actual: {actual:.2f}")


Predictions vs Actual:
Predicted: 1611.02, Actual: 1470.00
Predicted: 832.22, Actual: 750.00
Predicted: 1210.20, Actual: 870.75
Predicted: 884.22, Actual: 660.00
Predicted: 1966.05, Actual: 1319.25
Predicted: 876.82, Actual: 515.00
Predicted: 1394.90, Actual: 1475.75
Predicted: 1563.86, Actual: 1395.00
Predicted: 627.99, Actual: 606.50
Predicted: 749.57, Actual: 633.25
Predicted: 738.23, Actual: 776.50
Predicted: 846.42, Actual: 900.50
Predicted: 734.74, Actual: 824.25
Predicted: 886.72, Actual: 1406.00
Predicted: 940.62, Actual: 930.00
Predicted: 834.95, Actual: 769.50
Predicted: 946.74, Actual: 757.00
Predicted: 1288.52, Actual: 1443.75
Predicted: 773.67, Actual: 495.00
Predicted: 779.31, Actual: 1270.00
Predicted: 768.23, Actual: 928.75
Predicted: 1489.84, Actual: 1510.00
Predicted: 778.64, Actual: 625.00
Predicted: 1010.47, Actual: 787.50
Predicted: 787.59, Actual: 760.00
Predicted: 987.03, Actual: 1395.00
Predicted: 1381.27, Actual: 1030.00
Predicted: 726.16, Actual: 598.50
Predi

In [35]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test_orig, y_pred)
print(r2)

-0.050896644592285156


In [ ]:
# Very Bad!